In [58]:
import sys
from importlib import reload
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

skip_processing = False

# Ensure the notebook imports the local src tree, not an installed copy.
src_dir = Path.cwd() / "src"
if str(src_dir) in sys.path:
    sys.path.remove(str(src_dir))
sys.path.insert(0, str(src_dir))

import data_labeling_pipeline as dlp
import data_labeling_pipeline.hyperparameters as hyperparameters

reload(dlp)
reload(hyperparameters)

from data_labeling_pipeline.grandtour_missions import missions

In [59]:
from typing import Set

grand_tour_folder = Path("/datahome/lmertens/grand_tour_dataset")
output_root = Path("./output")
graphs_root = output_root / "graphs"
graphs_root.mkdir(parents=True, exist_ok=True)

all_mission_short_names: Set[str] = set([m["short_name"] for m in missions])
mission_short_names_to_process: Set[str] = set(all_mission_short_names)
# Example subset:
# mission_short_names_to_process = {"ETH-1", "SNOW-2"}

missions_to_process = [
    mission
    for mission in missions
    if mission["short_name"] in mission_short_names_to_process
]

print(f"Configured {len(missions_to_process)} missions for processing.")

Configured 49 missions for processing.


In [60]:
from data_labeling_pipeline import DataLabelingPipeline

if not skip_processing:
    processed_mission_outputs = []

    for mission in missions_to_process:
        mission_short_name = mission["short_name"]
        mission_path = grand_tour_folder / "ros1" / mission_short_name
        if not mission_path.exists():
            print(f"Skipping {mission_short_name}: missing {mission_path}")
            continue

        mission_output_dir = output_root / mission_short_name
        print(f"Processing {mission_short_name} -> {mission_output_dir}")
        pipeline = DataLabelingPipeline(mission_path=mission_path)
        pipeline.process_mission(output_dir=mission_output_dir)
        processed_mission_outputs.append({
            "mission": mission,
            "mission_path": mission_path,
            "output_dir": mission_output_dir,
        })

    print(f"Processed {len(processed_mission_outputs)} missions.")

Processing ETH-1 -> output/ETH-1
Using hyperparameters:
P_x=
[[1.5 0.  0. ]
 [0.  1.5 0. ]
 [0.  0.  1. ]]
P_m=
[[0.2 0.  0. ]
 [0.  0.2 0. ]
 [0.  0.  1.5]]
P_w=
[[1.2 0.  0. ]
 [0.  1.2 0. ]
 [0.  0.  1. ]]
N=10
STEP_SIZE=9
dt=0.1
Loading data from /datahome/lmertens/grand_tour_dataset/ros1/ETH-1 ...
Median dt: 0.050s, regular diffs: 5160/5190
Total synchronized steps: 2729
Running Receding Horizon Estimator ...
Processed step 0/2719
Processed step 900/2719
Processed step 1800/2719
Processed step 2700/2719
Saving continuous labels.
Mission processing complete.
Processing ETH-2 -> output/ETH-2
Using hyperparameters:
P_x=
[[1.5 0.  0. ]
 [0.  1.5 0. ]
 [0.  0.  1. ]]
P_m=
[[0.2 0.  0. ]
 [0.  0.2 0. ]
 [0.  0.  1.5]]
P_w=
[[1.2 0.  0. ]
 [0.  1.2 0. ]
 [0.  0.  1. ]]
N=10
STEP_SIZE=9
dt=0.1
Loading data from /datahome/lmertens/grand_tour_dataset/ros1/ETH-2 ...
Median dt: 0.050s, regular diffs: 5715/5738


ValueError: Expected `quat` to have shape (..., 4), got (0,).

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (13, 7),
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 150,
})

CONFIDENCE_THRESHOLD = 0.8
VIOLIN_POINTS = 50
VIOLIN_BW_METHOD = 0.15


def discover_processed_missions(output_root: Path):
    """
    Discover all processed missions by scanning output_root for subdirectories
    that contain the expected .npy output files.
    Returns a list of (mission_short_name, output_dir) tuples.
    """
    processed = []
    if not output_root.exists():
        return processed

    for subdir in output_root.iterdir():
        if not subdir.is_dir():
            continue
        # Skip the 'graphs' directory itself
        if subdir.name == "graphs":
            continue
        # Check if this directory has the expected outputs
        if (subdir / "trav_labels.npy").exists() and (
            subdir / "timestamp.npy"
        ).exists():
            processed.append((subdir.name, subdir))

    return sorted(processed)


def load_mission_statistics(output_dir: Path, mission_short_name: str):
    timestamp_path = output_dir / "timestamp.npy"
    labels_path = output_dir / "trav_labels.npy"
    mu_conf_path = output_dir / "mu_confidences.npy"
    nu_conf_path = output_dir / "nu_confidences.npy"

    if not timestamp_path.exists() or not labels_path.exists():
        print(
            f"Skipping {mission_short_name}: missing analysis outputs in {output_dir}"
        )
        return None

    trav_labels = np.load(labels_path)
    mu_confidences = np.load(mu_conf_path) if mu_conf_path.exists() else np.array([])
    nu_confidences = np.load(nu_conf_path) if nu_conf_path.exists() else np.array([])

    mu_labels = trav_labels[:, 0]
    nu_labels = trav_labels[:, 1]

    high_conf_mu = (
        mu_labels[mu_confidences > CONFIDENCE_THRESHOLD]
        if mu_confidences.size
        else np.array([])
    )
    high_conf_nu = (
        nu_labels[nu_confidences > CONFIDENCE_THRESHOLD]
        if nu_confidences.size
        else np.array([])
    )
    combined_confidences = (
        np.concatenate([arr for arr in [mu_confidences, nu_confidences] if arr.size])
        if (mu_confidences.size or nu_confidences.size)
        else np.array([])
    )

    return {
        "mission_short_name": mission_short_name,
        "high_conf_mu": high_conf_mu,
        "high_conf_nu": high_conf_nu,
        "mu_confidences": mu_confidences,
        "nu_confidences": nu_confidences,
        "combined_confidences": combined_confidences,
    }


def concatenate_non_empty(arrays):
    non_empty = [array for array in arrays if array.size]
    if not non_empty:
        return np.array([])
    return np.concatenate(non_empty)


def save_figure(fig, filename: str):
    path = graphs_root / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {path}")


def plot_distribution(values, title: str, xlabel: str, filename: str, color: str):
    if not values.size:
        print(f"Skipping {filename}: no values available.")
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    bins = np.linspace(0.0, 1.0, 41)
    ax.hist(values, bins=bins, density=True, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(
        float(np.mean(values)),
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"mean={np.mean(values):.3f}",
    )
    ax.axvline(
        float(np.median(values)),
        color="tab:orange",
        linestyle=":",
        linewidth=1.8,
        label=f"median={np.median(values):.3f}",
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_xlim(0.0, 1.0)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    save_figure(fig, filename)


def plot_confidence_distribution(values, filename: str):
    plot_distribution(
        values,
        title="Confidence Distribution Across the Entire Dataset",
        xlabel="Confidence",
        filename=filename,
        color="tab:purple",
    )


def plot_mission_confidence_distributions(stats_list, filename: str):
    if not stats_list:
        print(f"Skipping {filename}: no mission statistics available.")
        return

    bins = np.linspace(0.0, 1.0, 41)
    n_missions = len(stats_list)
    fig, axes = plt.subplots(
        n_missions,
        2,
        figsize=(12, max(3, n_missions * 1.2)),
        sharex=True,
        sharey=False,
    )

    if n_missions == 1:
        axes = np.array([axes])

    for row_idx, stats in enumerate(stats_list):
        name = stats["mission_short_name"]
        mu = np.asarray(stats.get("mu_confidences", np.array([])))
        nu = np.asarray(stats.get("nu_confidences", np.array([])))

        if mu.size == 0 and nu.size == 0:
            print(f"Skipping {name}: no confidence data.")
            continue

        axes[row_idx, 0].hist(
            mu, bins=bins, color="tab:blue", alpha=0.75, edgecolor="white"
        )
        axes[row_idx, 0].set_xlim(0.0, 1.0)
        axes[row_idx, 0].set_ylabel(name)
        axes[row_idx, 0].set_title("mu confidences" if row_idx == 0 else "")

        axes[row_idx, 1].hist(
            nu, bins=bins, color="tab:orange", alpha=0.75, edgecolor="white"
        )
        axes[row_idx, 1].set_xlim(0.0, 1.0)
        axes[row_idx, 1].set_title("nu confidences" if row_idx == 0 else "")

    axes[-1, 0].set_xlabel("Confidence")
    axes[-1, 1].set_xlabel("Confidence")
    fig.suptitle("Confidence Distributions per Mission", y=1.02)
    fig.tight_layout()
    save_figure(fig, filename)


def plot_mission_confidence_split_violin(stats_list, filename: str):
    if not stats_list:
        print(f"Skipping {filename}: no mission statistics available.")
        return

    import matplotlib.patches as patches
    from matplotlib.lines import Line2D

    mission_names = [stats["mission_short_name"] for stats in stats_list]
    positions = np.arange(len(mission_names))

    fig, ax = plt.subplots(figsize=(12, max(4, len(mission_names) * 0.5)))
    width = 0.8

    for idx, stats in enumerate(stats_list):
        mu = np.asarray(stats.get("mu_confidences", np.array([])))
        nu = np.asarray(stats.get("nu_confidences", np.array([])))
        pos = positions[idx]

        if mu.size:
            parts_mu = ax.violinplot(
                [mu],
                positions=[pos],
                vert=False,
                widths=width,
                points=VIOLIN_POINTS,
                bw_method=VIOLIN_BW_METHOD,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )
            for body in parts_mu["bodies"]:
                body.set_facecolor("tab:blue")
                body.set_alpha(0.7)
                clip = patches.Rectangle(
                    (0, pos), 1.0, width / 2.0, transform=ax.transData
                )
                body.set_clip_path(clip)

        if nu.size:
            parts_nu = ax.violinplot(
                [nu],
                positions=[pos],
                vert=False,
                widths=width,
                points=VIOLIN_POINTS,
                bw_method=VIOLIN_BW_METHOD,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )
            for body in parts_nu["bodies"]:
                body.set_facecolor("tab:orange")
                body.set_alpha(0.7)
                clip = patches.Rectangle(
                    (0, pos - width / 2.0), 1.0, width / 2.0, transform=ax.transData
                )
                body.set_clip_path(clip)

    ax.set_yticks(positions)
    ax.set_yticklabels(mission_names)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("Confidence")
    ax.set_title("Confidence split by mission")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(
        handles=[
            Line2D([0], [0], color="tab:blue", lw=6, label="mu confidence, upper half"),
            Line2D(
                [0], [0], color="tab:orange", lw=6, label="nu confidence, lower half"
            ),
        ],
        loc="best",
    )
    save_figure(fig, filename)

In [ ]:
processed_statistics = []

discovered_missions = discover_processed_missions(output_root)
print(f"Discovered {len(discovered_missions)} processed missions in {output_root}")

for mission_short_name, mission_output_dir in discovered_missions:
    stats = load_mission_statistics(mission_output_dir, mission_short_name)
    if stats is not None:
        processed_statistics.append(stats)

all_high_conf_mu = concatenate_non_empty([
    stats["high_conf_mu"] for stats in processed_statistics
])
all_high_conf_nu = concatenate_non_empty([
    stats["high_conf_nu"] for stats in processed_statistics
])
all_confidences = concatenate_non_empty([
    stats["combined_confidences"] for stats in processed_statistics
])

plot_distribution(
    all_high_conf_mu,
    title=f"Linear Traversability Distribution for Confidence > {CONFIDENCE_THRESHOLD:.1f}",
    xlabel="Linear traversability coefficient (mu)",
    filename="linear_traversability_high_confidence_distribution.pdf",
    color="tab:blue",
)

plot_distribution(
    all_high_conf_nu,
    title=f"Angular Traversability Distribution for Confidence > {CONFIDENCE_THRESHOLD:.1f}",
    xlabel="Angular traversability coefficient (nu)",
    filename="angular_traversability_high_confidence_distribution.pdf",
    color="tab:red",
)

plot_confidence_distribution(
    all_confidences,
    filename="confidence_distribution_all_missions.pdf",
)

plot_mission_confidence_distributions(
    processed_statistics,
    filename="confidence_distributions_by_mission.pdf",
)

plot_mission_confidence_split_violin(
    processed_statistics,
    filename="confidence_split_violin_by_mission.pdf",
)

Discovered 2 processed missions in output
Saved output/graphs/linear_traversability_high_confidence_distribution.pdf
Saved output/graphs/angular_traversability_high_confidence_distribution.pdf
Saved output/graphs/confidence_distribution_all_missions.pdf
Saved output/graphs/confidence_distributions_by_mission.pdf
Saved output/graphs/confidence_split_violin_by_mission.pdf
